In [ ]:
import os

persist_path = os.path.join(os.path.dirname(os.getcwd()), "store", "chroma_db")

print(persist_path)

print(os.path.abspath(os.path.join(os.path.dirname(os.getcwd()), "..")))

print(os.path.exists(persist_path) and os.listdir(persist_path))

In [ ]:
import os
from dotenv import load_dotenv

from retriever import get_retriever
from ingestion import load_documents
from chunking import split_documents
from rag_chain import build_rag_chain
from embedding import get_embeddings_provider
from vectordb import sync_documents_to_vector_db
from native_memory import add_memory_to_rag_chain

load_dotenv()

path = os.path.abspath(os.path.join(os.getcwd(), "..", "docs", "1706.03762v7.pdf"))

embedding_provider = get_embeddings_provider("openai")

docs = load_documents(path=path)
chunks = split_documents(documents=docs, chunk_size=500, chunk_overlap=100)
vectordb = sync_documents_to_vector_db(documents=chunks, embedding_engine=embedding_provider)
retriever = get_retriever(vectordb=vectordb, provider_name="openai")
rag_chain = build_rag_chain(provider_name="openai", retriever=retriever)
rag_chain_with_memory = add_memory_to_rag_chain(rag_chain=rag_chain, session_id="example", provider="openai", enabled=True)

response = rag_chain_with_memory.invoke(
    {"question": "Explain Attention is all you need in 2-3 sentences"},
    config={"configurable": {"session_id": "example", "provider": "openai"}}
)

print(f"\n\n{response.content}\n\n")
